# 06 · Errors & Context Managers

Full `try`/`except`/`else`/`finally`, `raise`, custom exception classes, writing context managers with `__enter__`/`__exit__`, and file handling. The `with` statement and `file I/O` are the same idea.

### 6.1 — Full `try`/`except`/`else`/`finally`

- `try`: the risky code
- `except`: runs on a matching error
- `else`: runs only if NO exception occurred
- `finally`: runs always (cleanup), exception or not

Write `safe_divide(a, b)` returning the quotient, or the string `"error"` on division by zero. Use 
`else` to return the result and `except` to return `"error"`.

In [1]:
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        return "error"
    else:
        return result

# Tests
assert safe_divide(10, 2) == 5
assert safe_divide(5, 0) == "error"
print("6.1 passed")

6.1 passed


### 6.2 — Raising exceptions

Use `raise` to signal an error yourself. Write `set_age(age)` that returns the age if it's a 
non-negative number, but raises `ValueError` if it's negative.

In [2]:
def set_age(age):
    if age >= 0:
        return age
    else:
        raise ValueError

# Tests
assert set_age(25) == 25
assert set_age(0) == 0
raised = False
try:
    set_age(-1)
except ValueError:
    raised = True
assert raised
print("6.2 passed")

6.2 passed


### 6.3 — Custom exception classes

A custom exception is just a class inheriting from `Exception`. It makes your errors specific and 
catchable by type. Define `InsufficientFundsError(Exception)`, then write `withdraw(balance, amount)` 
that raises it when `amount > balance`, otherwise returns the new balance.

In [3]:
class InsufficientFundsError(Exception):
    pass

def withdraw(balance, amount):
    if amount > balance:
        raise InsufficientFundsError()
    return balance - amount

# Tests
assert withdraw(100, 30) == 70
caught = False
try:
    withdraw(50, 100)
except InsufficientFundsError:
    caught = True
assert caught
print("6.3 passed")

6.3 passed


### 6.4 — Writing a context manager with a class

The `with` statement guarantees setup/teardown via `__enter__` (runs on entry, its return value is 
bound by `as`) and `__exit__` (runs on exit, even if an exception occurred). This is what 
`with open(...)` uses to always close the file.

Build a `Timer`-style `Tracker` that appends `"enter"` to a shared log on entry and `"exit"` on exit.

In [4]:
class Tracker:
    def __init__(self, log):
        self.log = log

    def __enter__(self):
        self.log.append("enter")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.log.append("exit")

# Tests
log = []
with Tracker(log):
    log.append("inside")
assert log == ["enter", "inside", "exit"]

# __exit__ runs even when the body raises
log2 = []
try:
    with Tracker(log2):
        raise ValueError("boom")
except ValueError:
    pass
assert log2 == ["enter", "exit"]
print("6.4 passed")

6.4 passed


### 6.5 — File handling (round trip)

`with open(path, "w")` writes; `with open(path, "r")` reads. The `with` block closes the file 
automatically (that's the context manager from 6.4 in action).

Write `save_and_load(path, lines)` that writes each string in `lines` to `path` (one per line), then 
reads the file back and returns the list of lines **without** trailing newlines.

In [8]:
def save_and_load(path, lines):
    with open(path, "w") as f:
        f.write("\n".join(lines))
    with open(path, "r") as f:
        return [line.rstrip("\n") for line in f]

# Tests
import os
path = "barkeep_test_io.txt"
result = save_and_load(path, ["alpha", "beta", "gamma"])
assert result == ["alpha", "beta", "gamma"]
assert save_and_load(path, []) == []
os.remove(path)
print("6.5 passed")

6.5 passed
